In [ ]:
import asyncio, json, logging, random, re, time
import nest_asyncio
nest_asyncio.apply()

from curl_cffi.requests import AsyncSession
from selectolax.parser import HTMLParser

# ── logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
logging.getLogger("curl_cffi").setLevel(logging.WARNING)

# ── proxy ─────────────────────────────────────────────────────────────────────
PROXY_HOST = "brd.superproxy.io"
PROXY_PORT  = 22225
PROXY_USER  = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS  = "ymg5cg3a1z33"

def make_proxies():
    token = f"{random.random():.16f}"
    user  = f"{PROXY_USER}-session-{token}"
    url   = f"http://{user}:{PROXY_PASS}@{PROXY_HOST}:{PROXY_PORT}"
    return {"http": url, "https": url}

BASE_URL = "https://www.jacob.de"
SORT     = "preis_up"
MAX_PAGE = 500

HEADERS = {
    "accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "de-DE,de;q=0.9,en;q=0.8",
    "referer":         "https://www.google.com/",
}

EXCLUDE_TOP = {"notebook sale", "jacob sale", "free shipping aktion"}

# ── fetch ─────────────────────────────────────────────────────────────────────
async def fetch(session: AsyncSession, url: str, retries: int = 3) -> tuple[str, int, float]:
    last_err = None
    for attempt in range(retries + 1):
        t0 = time.perf_counter()
        try:
            r = await session.get(
                url,
                headers=HEADERS,
                proxies=make_proxies(),
                timeout=45,
            )
            elapsed = time.perf_counter() - t0
            if r.status_code in (429, 500, 502, 503, 504):
                raise Exception(f"retryable {r.status_code}")
            r.raise_for_status()
            log.info("← %d  %5.1fs  %6d B  %s",
                     r.status_code, elapsed, len(r.content), url)
            return r.text, len(r.content), elapsed
        except Exception as e:
            elapsed = time.perf_counter() - t0
            log.warning("attempt %d FAILED  %.1fs  %s  %s",
                        attempt + 1, elapsed, type(e).__name__, url)
            last_err = e
            if attempt < retries:
                await asyncio.sleep(1.5 * (2 ** attempt))
    raise last_err

# ── category tree ─────────────────────────────────────────────────────────────
def normalize_url(url: str) -> str:
    from urllib.parse import urlparse
    p = urlparse(url)
    return f"{p.scheme}://{p.netloc}{p.path.rstrip('/')}/"

def is_category_url(url: str) -> bool:
    return (
        url.startswith(BASE_URL)
        and "/produkte/" not in url
        and "/q/" not in url
    )

def clean_label(text: str) -> str:
    return re.sub(r"\s*\d[\d\.\s]*$", "", text).strip()

def extract_top_links(html: str) -> list[tuple[str, str]]:
    from urllib.parse import urljoin
    tree = HTMLParser(html)
    nav  = tree.css_first("ul.nav__sub")
    if not nav:
        log.warning("extract_top_links: ul.nav__sub not found")
        return []
    seen = {}
    for a in nav.css("a[href]"):
        text = clean_label(a.text(strip=True))
        if not text or text.lower() in EXCLUDE_TOP:
            continue
        href = urljoin(BASE_URL, a.attributes["href"])
        if is_category_url(href):
            seen[normalize_url(href)] = text
    log.info("extract_top_links: %d unique top categories", len(seen))
    return [(t, u) for u, t in seen.items()]

def extract_child_links(html: str, current_url: str) -> list[tuple[str, str]]:
    from urllib.parse import urljoin
    tree = HTMLParser(html)
    cur  = normalize_url(current_url)

    nav = tree.css_first("li.nav__item.nav__current > ul.nav__sub")
    if not nav:
        navs = tree.css("ul.nav__sub")
        nav  = navs[-1] if navs else None
        if nav:
            for a in nav.css("a[href]"):
                if normalize_url(urljoin(BASE_URL, a.attributes.get("href", ""))) == cur:
                    return []

    if not nav:
        return []

    seen = {}
    for a in nav.css("a[href]"):
        text = clean_label(a.text(strip=True))
        href = urljoin(BASE_URL, a.attributes["href"])
        if text and is_category_url(href):
            seen[normalize_url(href)] = text
    return [(t, u) for u, t in seen.items()]

async def build_category_tree(
    session: AsyncSession,
    limit_top: int = None,
    max_depth: int = 4,
    sem: asyncio.Semaphore = None,
) -> list[dict]:
    html, _, _ = await fetch(session, BASE_URL)
    top_links  = extract_top_links(html)
    if limit_top:
        top_links = top_links[:limit_top]
    log.info("building tree from %d top categories", len(top_links))

    nodes   = {u: {"name": n, "url": u, "children": []} for n, u in top_links}
    visited = set()
    queue   = [(u, 0) for _, u in top_links]

    async def crawl_one(url: str, depth: int):
        async with sem:
            html, _, _ = await fetch(session, url)
        return url, depth, extract_child_links(html, url)

    while queue:
        batch   = [(u, d) for u, d in queue if u not in visited and d <= max_depth]
        visited |= {u for u, _ in batch}
        queue    = []
        if not batch:
            break
        results = await asyncio.gather(
            *[crawl_one(u, d) for u, d in batch],
            return_exceptions=True,
        )
        for res in results:
            if isinstance(res, Exception):
                log.warning("tree crawl error: %s", res)
                continue
            url, depth, children = res
            parent = nodes.setdefault(url, {"name": url, "url": url, "children": []})
            for cname, curl in children:
                if curl not in nodes:
                    nodes[curl] = {"name": cname, "url": curl, "children": []}
                child_node = nodes[curl]
                if child_node not in parent["children"]:
                    parent["children"].append(child_node)
                if curl not in visited and depth + 1 <= max_depth:
                    queue.append((curl, depth + 1))

    return [nodes[u] for _, u in top_links]

def build_breadcrumb_index(tree: list[dict]) -> dict[str, list[str]]:
    index = {}
    def walk(node, trail):
        new_trail = trail + ([node["name"]] if node.get("name") else [])
        if node.get("url"):
            index[normalize_url(node["url"])] = new_trail
        for child in node.get("children", []):
            walk(child, new_trail)
    for root in tree:
        walk(root, [])
    return index

# ── PLP ───────────────────────────────────────────────────────────────────────
def build_plp_url(cat_url: str, min_p: int, max_p: int, page: int) -> str:
    return f"{cat_url}?sortBy={SORT}&price-min={min_p}&price-max={max_p}&page={page}"

def plp_has_products(html: str) -> bool:
    return bool(HTMLParser(html).css("a[href*='/produkte/']"))

def extract_plp_items(html: str, breadcrumbs: list[str]) -> list[dict]:
    from urllib.parse import urljoin
    tree  = HTMLParser(html)
    items = {}

    # probe selectors so we can see which fires
    for sel in ["div.product-list__item", "article.product-item",
                "li.product-list__item", "div.product-box"]:
        hits = tree.css(sel)
        if hits:
            log.info("  PLP selector %-35s → %d hits", repr(sel), len(hits))

    for a in tree.css("a[href*='/produkte/']"):
        href = a.attributes.get("href", "")
        url  = href if href.startswith("http") else urljoin(BASE_URL, href)

        # walk up to find a container with a name element
        # name is NOT the link text (that's the price bug) — look for sibling/parent text nodes
        name = None
        parent = a.parent
        if parent:
            # try known name selectors within parent container
            for name_sel in [
                "div.product-list__name",
                "span.product-name",
                "p.product-name",
                "h2", "h3", "h4",
            ]:
                n = parent.css_first(name_sel)
                if n:
                    name = n.text(strip=True)
                    break

        # artnr from url slug
        artnr_m = re.search(r"artnr-(\d+)", url)
        artnr   = artnr_m.group(1) if artnr_m else None

        # price — look within same container
        price = None
        if parent:
            for price_sel in ["span.price__amount", "span.price", "div.price"]:
                pn = parent.css_first(price_sel)
                if pn:
                    pt = pn.text(strip=True)
                    pm = re.search(r"([\d\.]+,\d{2})\s*€", pt)
                    if pm:
                        price = pm.group(1)
                    break

        if url not in items:
            items[url] = {
                "url":         url,
                "name":        name,       # may be None — PDP will fill it
                "price":       price,
                "currency":    "EUR" if price else None,
                "artnr":       artnr,
                "breadcrumbs": breadcrumbs,
            }

    return list(items.values())

async def find_price_ceiling(
    session: AsyncSession, cat_url: str, sem: asyncio.Semaphore
) -> int:
    high = 10_000
    while high <= 2_000_000:
        async with sem:
            html, _, _ = await fetch(session, build_plp_url(cat_url, 1, high, 1))
        if not plp_has_products(html):
            break
        high *= 2
    log.info("price ceiling for %s: %d", cat_url, high)
    return high

async def split_ranges(
    session: AsyncSession,
    cat_url: str,
    min_p: int,
    max_p: int,
    sem: asyncio.Semaphore,
    depth: int = 0,
    max_depth: int = 20,
) -> list[tuple[int, int]]:
    if depth > max_depth or min_p >= max_p - 1:
        return [(min_p, max_p)]
    async with sem:
        html, _, _ = await fetch(session, build_plp_url(cat_url, min_p, max_p, MAX_PAGE))
    if not plp_has_products(html):
        return [(min_p, max_p)]
    mid = max(min_p + 1, (min_p + max_p) // 2)
    left, right = await asyncio.gather(
        split_ranges(session, cat_url, min_p, mid, sem, depth + 1),
        split_ranges(session, cat_url, mid, max_p, sem, depth + 1),
    )
    return left + right

async def collect_category(
    session: AsyncSession,
    cat_url: str,
    breadcrumbs: list[str],
    sem: asyncio.Semaphore,
) -> tuple[list[dict], int, float]:
    ceiling = await find_price_ceiling(session, cat_url, sem)

    async with sem:
        html, _, _ = await fetch(session, build_plp_url(cat_url, 1, ceiling, MAX_PAGE))
    ranges = (
        await split_ranges(session, cat_url, 1, ceiling, sem)
        if plp_has_products(html)
        else [(1, ceiling)]
    )
    log.info("category=%s  ranges=%d", cat_url, len(ranges))

    all_items  = []
    seen       = set()
    total_reqs = 0
    total_bytes = 0

    for min_p, max_p in ranges:
        page = 1
        while True:
            async with sem:
                html, nb, _ = await fetch(
                    session, build_plp_url(cat_url, min_p, max_p, page)
                )
            total_reqs  += 1
            total_bytes += nb
            items = extract_plp_items(html, breadcrumbs)
            if not items:
                break
            new = [it for it in items if it["url"] not in seen]
            seen |= {it["url"] for it in new}
            all_items.extend(new)
            log.info("  range=%d-%d  page=%d  new=%d  total=%d",
                     min_p, max_p, page, len(new), len(all_items))
            page += 1

    return all_items, total_reqs, total_bytes

# ── PDP ───────────────────────────────────────────────────────────────────────
def extract_jsonld_product(html: str) -> dict | None:
    blocks = re.findall(
        r'<script[^>]+type=["\']application/ld\+json["\'][^>]*>(.*?)</script>',
        html, flags=re.S | re.I,
    )
    for block in blocks:
        try:
            obj = json.loads(block.strip())
        except json.JSONDecodeError:
            continue
        items = obj if isinstance(obj, list) else obj.get("@graph", [obj])
        for item in items:
            if isinstance(item, dict) and item.get("@type") == "Product":
                return item
    return None

def parse_pdp(html: str, url: str, plp_data: dict) -> dict:
    product = extract_jsonld_product(html)
    condition_options = extract_condition_options(html, url)
    has_variable_pricing = len(condition_options) > 1

    new_option = next((x for x in condition_options if x["condition"] == "New"), None)
    refurbished_option = next((x for x in condition_options if x["condition"] == "Refurbished"), None)

    if not product:
        log.warning("no JSON-LD Product at %s", url)
        fallback_price = (
            (new_option or condition_options[0])["price"]
            if condition_options else plp_data.get("price")
        )
        fallback_currency = (
            (new_option or condition_options[0])["currency"]
            if condition_options else plp_data.get("currency")
        )

        if not new_option and fallback_price:
            new_option = {
                "condition": "New",
                "price": fallback_price,
                "currency": fallback_currency or "EUR",
                "url": url,
                "is_current": True,
            }

        return {
            "url":               url,
            "artnr":             plp_data.get("artnr"),
            "breadcrumbs":       plp_data.get("breadcrumbs", []),
            "name":              None,
            "brand":             None,
            "sku":               None,
            "mpn":               None,
            "ean":               None,
            "description":       None,
            "price":             fallback_price,
            "currency":          fallback_currency,
            "availability":      None,
            "category_path":     [],
            "condition_options": condition_options,
            "price_new":         new_option["price"] if new_option else "",
            "price_refurbished": refurbished_option["price"] if refurbished_option else "",
            "has_variable_pricing": has_variable_pricing,
            "pdp_parsed":        False,
        }

    brand_raw = product.get("brand")
    brand = (
        brand_raw.get("name") if isinstance(brand_raw, dict)
        else brand_raw if isinstance(brand_raw, str)
        else None
    )

    jsonld_price, currency, availability, offer_meta = extract_offer_prices(product)

    cat_path = []
    cat_raw  = product.get("category")
    if isinstance(cat_raw, str) and cat_raw.strip():
        cat_path = [c.strip() for c in cat_raw.split("/")]

    canonical_price = (
        new_option["price"] if new_option
        else condition_options[0]["price"] if condition_options
        else jsonld_price
        else plp_data.get("price")
    )
    canonical_currency = (
        new_option["currency"] if new_option
        else condition_options[0]["currency"] if condition_options
        else currency
        else plp_data.get("currency")
    )

    if not new_option and canonical_price:
        new_option = {
            "condition": "New",
            "price": canonical_price,
            "currency": canonical_currency or "EUR",
            "url": url,
            "is_current": True,
        }

    log.info(
        "pricing parse: variable=%s conditions=%d jsonld_type=%s min=%s max=%s url=%s",
        has_variable_pricing,
        len(condition_options),
        offer_meta.get("offer_type"),
        offer_meta.get("price_min"),
        offer_meta.get("price_max"),
        url,
    )

    return {
        "url":               url,
        "artnr":             plp_data.get("artnr"),
        "breadcrumbs":       plp_data.get("breadcrumbs", []),
        "name":              product.get("name"),
        "brand":             brand,
        "sku":               product.get("sku"),
        "mpn":               product.get("mpn"),
        "ean":               product.get("gtin13") or product.get("gtin14"),
        "description":       product.get("description"),
        "price":             canonical_price,
        "currency":          canonical_currency,
        "availability":      availability,
        "category_path":     cat_path,
        "condition_options": condition_options,
        "price_new":         new_option["price"] if new_option else "",
        "price_refurbished": refurbished_option["price"] if refurbished_option else "",
        "has_variable_pricing": has_variable_pricing,
        "jsonld_offer_type": offer_meta.get("offer_type"),
        "jsonld_price_min":  offer_meta.get("price_min"),
        "jsonld_price_max":  offer_meta.get("price_max"),
        "jsonld_offer_count": offer_meta.get("offer_count"),
        "pdp_parsed":        True,
    }

 

# ── discovery ─────────────────────────────────────────────────────────────────
async def discovery():
    sem = asyncio.Semaphore(5)

    async with AsyncSession(impersonate="chrome124") as session:

        # ── step 1: proxy health check ────────────────────────────────────────
        log.info("=== STEP 1: proxy health check ===")
        try:
            _, nb, t = await fetch(session, "https://geo.brdtest.com/welcome.txt")
            log.info("proxy OK  size=%d  time=%.2fs", nb, t)
        except Exception as e:
            log.critical("proxy FAILED — aborting: %s", e)
            return

        # ── step 2: category tree (2 tops for POC) ────────────────────────────
        log.info("=== STEP 2: category tree (limit_top=2) ===")
        tree  = await build_category_tree(session, limit_top=2, sem=sem)
        index = build_breadcrumb_index(tree)
        log.info("tree built: %d nodes", len(index))

        with open("jacob_category_tree.json", "w", encoding="utf-8") as f:
            json.dump(tree, f, ensure_ascii=False, indent=2)
        log.info("saved jacob_category_tree.json")

        # ── step 3: PLP for first category ────────────────────────────────────
        log.info("=== STEP 3: PLP sample ===")
        sample_url    = next(iter(index.keys()))
        sample_crumbs = index[sample_url]
        log.info("target: %s  crumbs=%s", sample_url, sample_crumbs)

        t0 = time.perf_counter()
        plp_items, plp_reqs, plp_bytes = await collect_category(
            session, sample_url, sample_crumbs, sem
        )
        plp_elapsed = time.perf_counter() - t0

        with open("jacob_plp_sample.json", "w", encoding="utf-8") as f:
            json.dump(plp_items, f, ensure_ascii=False, indent=2)
        log.info("saved jacob_plp_sample.json  items=%d", len(plp_items))

        # ── step 4: PDP for first 3 products ──────────────────────────────────
        log.info("=== STEP 4: PDP sample (3 products) ===")
        pdp_results = []
        pdp_sizes   = []
        pdp_times   = []
        sample_pdps = [it for it in plp_items if it.get("artnr")][:3]

        for item in sample_pdps:
            t0 = time.perf_counter()
            async with sem:
                html, nb, _ = await fetch(session, item["url"])
            pdp_sizes.append(nb)
            pdp_times.append(time.perf_counter() - t0)
            row = parse_pdp(html, item["url"], item)
            log.info("PDP: name=%-50s  ean=%s  price=%s %s",
                     (row.get("name") or "")[:50],
                     row.get("ean"), row.get("price"), row.get("currency"))
            pdp_results.append(row)

        with open("jacob_pdp_sample.json", "w", encoding="utf-8") as f:
            json.dump(pdp_results, f, ensure_ascii=False, indent=2)
        log.info("saved jacob_pdp_sample.json")

    # ── report ────────────────────────────────────────────────────────────────
    n              = len(plp_items)
    avg_pdp_kb     = sum(pdp_sizes) / len(pdp_sizes) / 1024 if pdp_sizes else 0
    avg_pdp_t      = sum(pdp_times) / len(pdp_times) if pdp_times else 0
    avg_plp_kb     = plp_bytes / max(plp_reqs, 1) / 1024
    avg_plp_t      = plp_elapsed / max(plp_reqs, 1)

    total_cats     = len(index)
    est_products   = n * total_cats           # rough: same density per cat
    est_plp_reqs   = plp_reqs * total_cats
    est_pdp_reqs   = est_products             # 1 PDP per product
    est_plp_gb     = (avg_plp_kb * 1024 * est_plp_reqs) / 1024**3
    est_pdp_gb     = (avg_pdp_kb * 1024 * est_pdp_reqs) / 1024**3
    est_total_gb   = est_plp_gb + est_pdp_gb

    name_missing   = sum(1 for it in plp_items if not it.get("name"))
    price_missing  = sum(1 for it in plp_items if not it.get("price"))

    pdp_ok         = sum(1 for p in pdp_results if p.get("pdp_parsed"))

    print(f"""
╔══════════════════════════════════════════════════════════════════╗
  jacob.de — DISCOVERY REPORT
╚══════════════════════════════════════════════════════════════════╝

STACK
  HTTP client       : curl_cffi chrome124
  Parser            : selectolax (PLP)  |  JSON-LD regex (PDP)
  Proxy             : Bright Data datacenter (DC)
  Concurrency       : asyncio.Semaphore(5)

CATEGORY TREE  (limit_top=2 for POC — full run removes this cap)
  Nodes discovered  : {total_cats}
  Saved             : jacob_category_tree.json

PLP SAMPLE  ({sample_url})
  Products found    : {n}
  Requests made     : {plp_reqs}
  Bytes transferred : {plp_bytes/1024:.1f} KB
  Avg page size     : {avg_plp_kb:.1f} KB
  Avg response time : {avg_plp_t:.2f}s
  Time elapsed      : {plp_elapsed:.1f}s
  Name missing      : {name_missing}/{n}  {"⚠ PLP name selector needs debug" if name_missing > n*0.5 else "✓"}
  Price missing     : {price_missing}/{n}  (expected — PDP fills this)

PDP SAMPLE  (3 products)
  Parsed OK         : {pdp_ok}/3
  Avg page size     : {avg_pdp_kb:.1f} KB
  Avg response time : {avg_pdp_t:.2f}s
  Fields captured   : name, brand, sku, mpn, ean,
                      description, price, currency,
                      availability, category_path
  Saved             : jacob_pdp_sample.json

STRATEGY
  PLP role          : URL + artnr discovery only (name unreliable from PLP)
  PDP role          : all core fields via JSON-LD — REQUIRED for every product
  Page cap strategy : price-range binary split (≤{MAX_PAGE} pages per range)
  Sort              : {SORT}
  Dedup key         : artnr (from URL slug)

FULL RUN ESTIMATES  (scaled from {total_cats} categories)
  Est. total products : ~{est_products:,}
  Est. PLP requests   : ~{est_plp_reqs:,}
  Est. PDP requests   : ~{est_pdp_reqs:,}
  Est. PLP bandwidth  : ~{est_plp_gb:.2f} GB
  Est. PDP bandwidth  : ~{est_pdp_gb:.2f} GB
  Est. TOTAL bandwidth: ~{est_total_gb:.2f} GB
  Note: estimates rough — based on {sample_url} density

OUTPUT FILES
  jacob_category_tree.json  category nav tree with breadcrumbs
  jacob_plp_sample.json     {n} product stubs from sample category
  jacob_pdp_sample.json     3 fully parsed products (core fields)

SAMPLE PRODUCTS
""")
    for p in pdp_results:
        print(f"  url          : {p['url']}")
        print(f"  name         : {p.get('name')}")
        print(f"  brand        : {p.get('brand')}")
        print(f"  mpn          : {p.get('mpn')}")
        print(f"  ean          : {p.get('ean')}")
        print(f"  price        : {p.get('price')} {p.get('currency')}")
        print(f"  availability : {p.get('availability')}")
        print(f"  category     : {' > '.join(p.get('category_path', []))}")
        print(f"  pdp_parsed   : {p.get('pdp_parsed')}")
        print()

asyncio.run(discovery())

16:01:36 [INFO] === STEP 1: proxy health check ===
16:01:41 [INFO] ← 200    4.2s     618 B  https://geo.brdtest.com/welcome.txt
16:01:41 [INFO] proxy OK  size=618  time=4.15s
16:01:41 [INFO] === STEP 2: category tree (limit_top=2) ===
16:01:42 [INFO] ← 200    1.5s  138723 B  https://www.jacob.de
16:01:42 [INFO] extract_top_links: 9 unique top categories
16:01:42 [INFO] building tree from 2 top categories
16:01:44 [INFO] ← 200    2.1s  605594 B  https://www.jacob.de/enterprise/
16:01:45 [INFO] ← 200    2.5s  411613 B  https://www.jacob.de/apple/
16:01:46 [INFO] ← 200    1.3s  364791 B  https://www.jacob.de/apple-iphone/
16:01:46 [INFO] ← 200    1.3s  191991 B  https://www.jacob.de/apple-displays/
16:01:46 [INFO] ← 200    1.5s  388144 B  https://www.jacob.de/apple-ipad-familie/
16:01:46 [INFO] ← 200    1.8s  353909 B  https://www.jacob.de/apple-watch/
16:01:46 [INFO] ← 200    1.8s  357072 B  https://www.jacob.de/apple-mac/
16:01:47 [INFO] ← 200    1.2s  168427 B  https://www.jacob.de/app


╔══════════════════════════════════════════════════════════════════╗
  jacob.de — DISCOVERY REPORT
╚══════════════════════════════════════════════════════════════════╝

STACK
  HTTP client       : curl_cffi chrome124
  Parser            : selectolax (PLP)  |  JSON-LD regex (PDP)
  Proxy             : Bright Data datacenter (DC)
  Concurrency       : asyncio.Semaphore(5)

CATEGORY TREE  (limit_top=2 for POC — full run removes this cap)
  Nodes discovered  : 99
  Saved             : jacob_category_tree.json

PLP SAMPLE  (https://www.jacob.de/apple/)
  Products found    : 1106
  Requests made     : 57
  Bytes transferred : 22969.9 KB
  Avg page size     : 403.0 KB
  Avg response time : 7.20s
  Time elapsed      : 410.7s
  Name missing      : 1106/1106  ⚠ PLP name selector needs debug
  Price missing     : 1106/1106  (expected — PDP fills this)

PDP SAMPLE  (3 products)
  Parsed OK         : 3/3
  Avg page size     : 194.5 KB
  Avg response time : 1.71s
  Fields captured   : name, brand, 

In [4]:
import asyncio, json, logging, random, re, time
import nest_asyncio
nest_asyncio.apply()

from curl_cffi.requests import AsyncSession
from selectolax.parser import HTMLParser
from urllib.parse import urljoin, urlparse

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
logging.getLogger("curl_cffi").setLevel(logging.WARNING)

# ── proxy ─────────────────────────────────────────────────────────────────────
PROXY_HOST = "brd.superproxy.io"
PROXY_PORT  = 22225
PROXY_USER  = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS  = "ymg5cg3a1z33"

# sticky session — one token per worker slot, reused across requests
# call make_sticky_proxies() once per worker and pass it through
def make_sticky_proxies(slot: int) -> dict:
    user = f"{PROXY_USER}-session-slot{slot}"
    url  = f"http://{user}:{PROXY_PASS}@{PROXY_HOST}:{PROXY_PORT}"
    return {"http": url, "https": url}

# slot pool — semaphore index maps to a fixed proxy session
CONCURRENCY  = 5
SLOT_PROXIES = [make_sticky_proxies(i) for i in range(CONCURRENCY)]

BASE_URL    = "https://www.jacob.de"
EXCLUDE_TOP = {"notebook sale", "jacob sale", "free shipping aktion"}
SORT        = "preis_up"
MAX_PAGE    = 500
AVG_PRODUCTS_PER_PAGE = 20    # jacob PLP shows ~20 products/page
AVG_PDP_SIZE_KB       = 194.5 # measured from our 3-product sample
AVG_PLP_SIZE_KB       = 403.0 # measured from apple category sample

HEADERS = {
    "accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "de-DE,de;q=0.9,en;q=0.8",
    "referer":         "https://www.google.com/",
}

# ── fetch with slot-based sticky proxy ───────────────────────────────────────
async def fetch(
    session: AsyncSession,
    url: str,
    sem: asyncio.Semaphore,
    slot: int = 0,
    retries: int = 3,
):
    proxies  = SLOT_PROXIES[slot % CONCURRENCY]
    last_err = None
    for attempt in range(retries + 1):
        t0 = time.perf_counter()
        try:
            r = await session.get(
                url,
                headers=HEADERS,
                proxies=proxies,
                timeout=45,
            )
            elapsed = time.perf_counter() - t0
            if r.status_code in (429, 500, 502, 503, 504):
                raise Exception(f"retryable {r.status_code}")
            r.raise_for_status()
            log.info("← %d  %5.1fs  %d bytes  [slot%d]  %s",
                     r.status_code, elapsed, len(r.content), slot, url)
            return r.text, len(r.content), elapsed
        except Exception as e:
            elapsed = time.perf_counter() - t0
            log.warning("slot%d  attempt %d FAILED  %.1fs  %s  %s",
                        slot, attempt + 1, elapsed, type(e).__name__, url)
            last_err = e
            if attempt < retries:
                await asyncio.sleep(2.0 * (2 ** attempt))
    raise last_err

# ── url helpers ───────────────────────────────────────────────────────────────
def normalize_url(url: str) -> str:
    p = urlparse(url)
    return f"{p.scheme}://{p.netloc}{p.path.rstrip('/')}/"

def is_category_url(url: str) -> bool:
    return (
        url.startswith(BASE_URL)
        and "/produkte/" not in url
        and "/q/" not in url
    )

def clean_label(text: str) -> str:
    return re.sub(r"\s*\d[\d\.\s]*$", "", text).strip()

# ── nav parsers ───────────────────────────────────────────────────────────────
def extract_top_links(html: str) -> list[tuple[str, str]]:
    tree = HTMLParser(html)
    nav  = tree.css_first("ul.nav__sub")
    if not nav:
        return []
    seen = {}
    for a in nav.css("a[href]"):
        text = clean_label(a.text(strip=True))
        if not text or text.lower() in EXCLUDE_TOP:
            continue
        href = urljoin(BASE_URL, a.attributes["href"])
        if is_category_url(href):
            seen[normalize_url(href)] = text
    return [(t, u) for u, t in seen.items()]

def extract_child_links(html: str, current_url: str) -> list[tuple[str, str]]:
    tree = HTMLParser(html)
    cur  = normalize_url(current_url)

    nav = tree.css_first("li.nav__item.nav__current > ul.nav__sub")
    if not nav:
        navs = tree.css("ul.nav__sub")
        nav  = navs[-1] if navs else None
        if nav:
            for a in nav.css("a[href]"):
                if normalize_url(urljoin(BASE_URL, a.attributes.get("href", ""))) == cur:
                    return []
    if not nav:
        return []

    seen = {}
    for a in nav.css("a[href]"):
        text = clean_label(a.text(strip=True))
        href = urljoin(BASE_URL, a.attributes["href"])
        if text and is_category_url(href):
            seen[normalize_url(href)] = text
    return [(t, u) for u, t in seen.items()]

# ── product count probe ───────────────────────────────────────────────────────
def extract_product_count(html: str) -> int | None:
    """Extract total product count from PLP page — e.g. '1.234 Artikel'"""
    tree = HTMLParser(html)
    for sel in [
        "span.result-count",
        "div.result-count",
        "p.result-count",
        "span.product-count",
        "div.listing__info",
        "h1",
        "title",
    ]:
        node = tree.css_first(sel)
        if node:
            text = node.text(strip=True)
            m = re.search(r"([\d\.]+)\s*(Artikel|Produkte|Ergebnisse|results)", text, re.I)
            if m:
                return int(m.group(1).replace(".", ""))
    # fallback: count product links on page 1
    items = tree.css("a[href*='/produkte/']")
    return len(items) if items else None

def extract_last_page(html: str) -> int | None:
    """Extract last page number from pagination."""
    tree = HTMLParser(html)
    last = 1
    for sel in ["a.pagination__link", "a[href*='page=']", "li.pagination__item a"]:
        for a in tree.css(sel):
            href = a.attributes.get("href", "")
            m    = re.search(r"page=(\d+)", href)
            if m:
                last = max(last, int(m.group(1)))
    return last if last > 1 else None

# ── category probe ────────────────────────────────────────────────────────────
async def probe_category(
    session: AsyncSession,
    url: str,
    crumbs: list[str],
    sem: asyncio.Semaphore,
    slot: int,
) -> dict:
    """
    Fetch page 1 of a category to extract product count + last page.
    Does NOT crawl all pages — pure estimation.
    """
    probe_url = f"{url}?sortBy={SORT}&price-min=1&price-max=9999999&page=1"
    try:
        async with sem:
            html, nb, t = await fetch(session, probe_url, sem=asyncio.Semaphore(999),
                                      slot=slot)
        count     = extract_product_count(html)
        last_page = extract_last_page(html)

        # if last_page not found, estimate from count
        if not last_page and count:
            last_page = max(1, -(-count // AVG_PRODUCTS_PER_PAGE))  # ceiling div

        # estimate pages needed with price-range split
        pages_needed = last_page or 1
        needs_split  = pages_needed > MAX_PAGE
        est_ranges   = max(1, -(-pages_needed // MAX_PAGE)) if needs_split else 1
        est_plp_reqs = est_ranges * min(pages_needed, MAX_PAGE)
        est_products = count or (pages_needed * AVG_PRODUCTS_PER_PAGE)

        est_plp_kb   = est_plp_reqs * AVG_PLP_SIZE_KB
        est_pdp_kb   = est_products * AVG_PDP_SIZE_KB
        est_total_kb = est_plp_kb + est_pdp_kb

        return {
            "url":           url,
            "breadcrumbs":   crumbs,
            "product_count": count,
            "last_page":     last_page,
            "needs_split":   needs_split,
            "est_ranges":    est_ranges,
            "est_plp_reqs":  est_plp_reqs,
            "est_pdp_reqs":  est_products,
            "est_total_reqs":est_plp_reqs + est_products,
            "est_plp_kb":    round(est_plp_kb),
            "est_pdp_kb":    round(est_pdp_kb),
            "est_total_kb":  round(est_total_kb),
            "probe_size_kb": round(nb / 1024, 1),
            "probe_time_s":  round(t, 2),
            "error":         None,
        }
    except Exception as e:
        log.warning("probe FAILED %s  %s", url, e)
        return {
            "url": url, "breadcrumbs": crumbs,
            "error": str(e),
            "est_total_reqs": 0, "est_total_kb": 0,
        }

# ── tree builder ──────────────────────────────────────────────────────────────
async def build_category_tree(
    session: AsyncSession,
    sem: asyncio.Semaphore,
    limit_top: int = None,
    max_depth: int = 4,
) -> list[dict]:
    html, _, _ = await fetch(session, BASE_URL, sem, slot=0)
    top_links   = extract_top_links(html)
    if limit_top:
        top_links = top_links[:limit_top]
    log.info("top-level categories: %d", len(top_links))

    nodes   = {u: {"name": n, "url": u, "children": []} for n, u in top_links}
    visited = set()
    queue   = [(u, 0) for _, u in top_links]
    slot_counter = 0

    while queue:
        # process in small batches to avoid burst
        batch    = []
        while queue and len(batch) < CONCURRENCY:
            u, d = queue.pop(0)
            if u not in visited and d <= max_depth:
                batch.append((u, d))
                visited.add(u)

        if not batch:
            continue

        async def crawl_one(url, depth, slot):
            html, _, _ = await fetch(session, url, sem, slot=slot)
            return url, depth, extract_child_links(html, url)

        results = await asyncio.gather(
            *[crawl_one(u, d, (slot_counter + i) % CONCURRENCY)
              for i, (u, d) in enumerate(batch)],
            return_exceptions=True,
        )
        slot_counter += len(batch)

        for res in results:
            if isinstance(res, Exception):
                log.warning("tree crawl error: %s", res)
                continue
            url, depth, children = res
            parent = nodes.setdefault(url, {"name": url, "url": url, "children": []})
            seen_ch = {c["url"] for c in parent["children"]}
            for cname, curl in children:
                if curl not in nodes:
                    nodes[curl] = {"name": cname, "url": curl, "children": []}
                if curl not in seen_ch:
                    parent["children"].append(nodes[curl])
                    seen_ch.add(curl)
                if curl not in visited and depth + 1 <= max_depth:
                    queue.append((curl, depth + 1))

        log.info("tree progress: visited=%d  queue=%d", len(visited), len(queue))

    return [nodes[u] for _, u in top_links]

def build_breadcrumb_index(tree: list[dict]) -> dict[str, list[str]]:
    index = {}
    def walk(node, trail):
        new_trail = trail + ([node["name"]] if node.get("name") else [])
        if node.get("url"):
            index[normalize_url(node["url"])] = new_trail
        for child in node.get("children", []):
            walk(child, new_trail)
    for root in tree:
        walk(root, [])
    return index

def flatten_tree(tree: list[dict]) -> list[dict]:
    out = []
    def walk(node):
        out.append(node)
        for child in node.get("children", []):
            walk(child)
    for root in tree:
        walk(root)
    return out

# ── main audit ────────────────────────────────────────────────────────────────
async def category_depth_audit():
    sem = asyncio.Semaphore(CONCURRENCY)
    t_start = time.perf_counter()

    async with AsyncSession(impersonate="chrome124") as session:

        # step 1: tree
        log.info("=== STEP 1: building category tree ===")
        tree  = await build_category_tree(session, sem, limit_top=None, max_depth=4)
        index = build_breadcrumb_index(tree)
        log.info("tree complete: %d nodes  %.1fs", len(index), time.perf_counter()-t_start)

        with open("jacob_category_tree_full.json", "w", encoding="utf-8") as f:
            json.dump(tree, f, ensure_ascii=False, indent=2)

        # step 2: probe every leaf category for product count + page estimate
        all_nodes   = flatten_tree(tree)
        node_by_url = {normalize_url(n["url"]): n for n in all_nodes if n.get("url")}

        leaf_entries = [
            (url, crumbs)
            for url, crumbs in index.items()
            if not node_by_url.get(url, {}).get("children")
        ]
        log.info("=== STEP 2: probing %d leaf categories ===", len(leaf_entries))

        # probe in batches of CONCURRENCY, slot-assigned
        probes = []
        for i in range(0, len(leaf_entries), CONCURRENCY):
            batch = leaf_entries[i:i + CONCURRENCY]
            results = await asyncio.gather(*[
                probe_category(session, url, crumbs, sem, slot=j % CONCURRENCY)
                for j, (url, crumbs) in enumerate(batch)
            ])
            probes.extend(results)
            await asyncio.sleep(0.5)   # gentle pause between batches

    # ── analysis ──────────────────────────────────────────────────────────────
    valid    = [p for p in probes if not p.get("error")]
    errored  = [p for p in probes if p.get("error")]

    depth_counts = {}
    outlet_nodes = []
    top_level    = {}

    for url, crumbs in index.items():
        d = len(crumbs)
        depth_counts[d] = depth_counts.get(d, 0) + 1
        if any(kw in url for kw in ("outlet", "sale", "aktion")):
            outlet_nodes.append((url, crumbs))
        top = crumbs[0] if crumbs else "unknown"
        if top not in top_level:
            top_level[top] = {"nodes": 0, "leaves": 0, "max_depth": 0,
                               "est_products": 0, "est_reqs": 0, "est_kb": 0}
        top_level[top]["nodes"]    += 1
        top_level[top]["max_depth"] = max(top_level[top]["max_depth"], d)
        node = node_by_url.get(url)
        if node and not node.get("children"):
            top_level[top]["leaves"] += 1

    for p in valid:
        top = p["breadcrumbs"][0] if p["breadcrumbs"] else "unknown"
        if top in top_level:
            top_level[top]["est_products"] += p.get("est_pdp_reqs", 0)
            top_level[top]["est_reqs"]     += p.get("est_total_reqs", 0)
            top_level[top]["est_kb"]       += p.get("est_total_kb", 0)

    total_products = sum(p.get("est_pdp_reqs", 0) for p in valid)
    total_reqs     = sum(p.get("est_total_reqs", 0) for p in valid)
    total_kb       = sum(p.get("est_total_kb", 0) for p in valid)
    needs_split    = [p for p in valid if p.get("needs_split")]

    # ── report ────────────────────────────────────────────────────────────────
    elapsed = time.perf_counter() - t_start
    print(f"""
╔══════════════════════════════════════════════════════════════════╗
  jacob.de — FULL CATEGORY AUDIT + TRAFFIC ESTIMATES
╚══════════════════════════════════════════════════════════════════╝

RUNTIME               : {elapsed:.0f}s
PROXY                 : DC sticky sessions (slot0–slot{CONCURRENCY-1})
CONCURRENCY           : {CONCURRENCY} workers

CATEGORY TREE
  Total nodes         : {len(index)}
  Leaf nodes (scrape) : {len(leaf_entries)}
  Probe errors        : {len(errored)}
  Outlet/sale nodes   : {len(outlet_nodes)}  ← post-dedup on artnr

DEPTH DISTRIBUTION
""")
    for d in sorted(depth_counts):
        bar = "█" * min(depth_counts[d], 50)
        print(f"  depth {d}  {depth_counts[d]:4d}  {bar}")

    print(f"""
FULL RUN ESTIMATES  (leaf categories only, PDP required for every product)
  Est. total products : {total_products:>10,}
  Est. total requests : {total_reqs:>10,}  (PLP + PDP)
  Est. total bandwidth: {total_kb/1024/1024:>10.2f} GB
  Categories needing price-range split: {len(needs_split)}

PER TOP-LEVEL CATEGORY
  {'Category':<38} {'Nodes':>6} {'Leaves':>7} {'MaxD':>5} {'~Products':>10} {'~Reqs':>8} {'~GB':>6}
  {'-'*83}""")
    for name, s in sorted(top_level.items(), key=lambda x: -x[1]["est_products"]):
        gb = s["est_kb"] / 1024 / 1024
        print(f"  {name:<38} {s['nodes']:>6} {s['leaves']:>7} {s['max_depth']:>5}"
              f" {s['est_products']:>10,} {s['est_reqs']:>8,} {gb:>6.2f}")

    print(f"""
PER LEAF CATEGORY  (sorted by est. products, top 40)
  {'Breadcrumb':<55} {'Prods':>6} {'PLPReqs':>8} {'PDPReqs':>8} {'KB':>8} {'Split':>6}
  {'-'*95}""")
    for p in sorted(valid, key=lambda x: -(x.get("est_pdp_reqs") or 0))[:40]:
        crumb = " > ".join(p["breadcrumbs"])[-54:]
        split = "YES" if p.get("needs_split") else "-"
        print(f"  {crumb:<55} {p.get('est_pdp_reqs',0):>6,}"
              f" {p.get('est_plp_reqs',0):>8,} {p.get('est_pdp_reqs',0):>8,}"
              f" {p.get('est_total_kb',0):>8,} {split:>6}")

    if errored:
        print(f"\nPROBE ERRORS ({len(errored)})")
        for p in errored:
            print(f"  {p['url']}  →  {p['error']}")

    with open("jacob_category_audit.json", "w", encoding="utf-8") as f:
        json.dump({
            "summary": {
                "total_nodes":    len(index),
                "leaf_nodes":     len(leaf_entries),
                "est_products":   total_products,
                "est_requests":   total_reqs,
                "est_bandwidth_gb": round(total_kb/1024/1024, 2),
            },
            "per_category": probes,
            "top_level":    {k: v for k, v in top_level.items()},
        }, f, ensure_ascii=False, indent=2)
    log.info("saved jacob_category_audit.json")

asyncio.run(category_depth_audit())

23:26:19 [INFO] === STEP 1: building category tree ===
23:26:21 [INFO] ← 200    1.5s  138723 bytes  [slot0]  https://www.jacob.de
23:26:21 [INFO] top-level categories: 9
23:26:21 [INFO] ← 200    0.4s  411610 bytes  [slot0]  https://www.jacob.de/apple/
23:26:22 [INFO] ← 200    1.3s  457695 bytes  [slot2]  https://www.jacob.de/haushalt/
23:26:22 [INFO] ← 200    1.4s  510621 bytes  [slot3]  https://www.jacob.de/netzwerk/
23:26:23 [INFO] ← 200    1.9s  574600 bytes  [slot4]  https://www.jacob.de/notebook-tablet/
23:26:23 [INFO] ← 200    2.2s  605593 bytes  [slot1]  https://www.jacob.de/enterprise/
23:26:23 [INFO] tree progress: visited=5  queue=83
23:26:24 [INFO] ← 200    0.5s  485115 bytes  [slot2]  https://www.jacob.de/smartphone-foto/
23:26:24 [INFO] ← 200    0.5s  357055 bytes  [slot4]  https://www.jacob.de/apple-mac/
23:26:24 [INFO] ← 200    0.5s  436810 bytes  [slot3]  https://www.jacob.de/tv-audio/
23:26:24 [INFO] ← 200    1.1s  451842 bytes  [slot0]  https://www.jacob.de/office/
23


╔══════════════════════════════════════════════════════════════════╗
  jacob.de — FULL CATEGORY AUDIT + TRAFFIC ESTIMATES
╚══════════════════════════════════════════════════════════════════╝

RUNTIME               : 359s
PROXY                 : DC sticky sessions (slot0–slot4)
CONCURRENCY           : 5 workers

CATEGORY TREE
  Total nodes         : 648
  Leaf nodes (scrape) : 554
  Probe errors        : 0
  Outlet/sale nodes   : 3  ← post-dedup on artnr

DEPTH DISTRIBUTION

  depth 1     9  █████████
  depth 2   129  ██████████████████████████████████████████████████
  depth 3   348  ██████████████████████████████████████████████████
  depth 4   140  ██████████████████████████████████████████████████
  depth 5    18  ██████████████████
  depth 6     4  ████

FULL RUN ESTIMATES  (leaf categories only, PDP required for every product)
  Est. total products :     26,406
  Est. total requests :     37,261  (PLP + PDP)
  Est. total bandwidth:       9.07 GB
  Categories needing price-range s

In [21]:
import asyncio, csv, json, re, time, logging
from playwright.async_api import async_playwright

log = logging.getLogger(__name__)

PRODUCTS_PER_PAGE = 20
CAP_THRESHOLD     = 500 * PRODUCTS_PER_PAGE  # 10,000
CONCURRENCY       = 5

with open("jacob_category_tree_full.json") as f:
    TREE = json.load(f)

def pw_proxy(slot: int) -> dict:
    return {
        "server":   f"http://{PROXY_HOST}:{PROXY_PORT}",
        "username": f"{PROXY_USER}-session-slot{slot}",
        "password": PROXY_PASS,
    }

def parse_int(s: str) -> int | None:
    clean = re.sub(r"[^\d]", "", s)
    return int(clean) if clean else None


async def fetch_all_category_counts(pw_page, url: str) -> dict:
    """
    Load any category page and return {name_normalized: count} from
    the active nav__current item's rendered text, which contains all
    child names and counts as plain text.
    Also works for /q/ which lists all top-level counts.
    """
    await pw_page.goto(url, wait_until="networkidle", timeout=30_000)

    # grab all nav__count spans with their link text (name + count together)
    items = await pw_page.eval_on_selector_all(
        "ul.nav__sub a.nav__link",
        """els => els.map(a => {
            const span = a.querySelector('span.nav__count');
            if (!span) return null;
            // clone to get just the text node (name without count)
            const clone = a.cloneNode(true);
            clone.querySelectorAll('span').forEach(s => s.remove());
            const name  = clone.innerText.trim();
            const count = parseInt(span.innerText.trim().replace(/\\./g,'').replace(/,/g,''), 10);
            return { name, count };
        }).filter(Boolean)"""
    )

    return {
        item["name"].lower().strip(): item["count"]
        for item in items
        if item and item["count"]
    }


async def probe_node(playwright, sem, slot, node, parent_url, depth=0) -> dict:
    indent = "  " * depth
    result = {
        "name":           node["name"],
        "url":            node["url"],
        "reported_count": None,
        "true_count":     None,
        "capped":         False,
        "drilled":        False,
        "children":       [],
    }

    async with sem:
        browser = await playwright.chromium.launch(headless=True, proxy=pw_proxy(slot))
        try:
            context = await browser.new_context(locale="de-DE")
            pw_page = await context.new_page()
            await pw_page.route("**/*", lambda route: (
                route.abort() if route.request.resource_type in
                ("image", "media", "font", "stylesheet") else route.continue_()
            ))

            nav = await fetch_all_category_counts(pw_page, parent_url)
            key = node["name"].lower().strip()
            raw_count = nav.get(key)

            if raw_count is not None:
                capped = raw_count >= CAP_THRESHOLD
                result.update({
                    "reported_count": raw_count,
                    "capped":         capped,
                })
                log.info("%s%-45s  reported=%-8d  %s",
                         indent, node["name"], raw_count,
                         "*** CAPPED ***" if capped else "ok")
            else:
                log.warning("%s%-45s  not found (nav had: %s)",
                            indent, node["name"],
                            list(nav.keys())[:5])

        except Exception as e:
            log.error("%s%-45s  ERROR: %s", indent, node["name"], e)
        finally:
            await browser.close()

    # drill into children if node has children in tree
    # always drill — even if not capped — to get true granular counts
    children_in_tree = node.get("children", [])
    if children_in_tree:
        result["drilled"] = True
        child_tasks = [
            probe_node(
                playwright, sem,
                slot=(slot + i) % CONCURRENCY,
                node=child,
                parent_url=node["url"],  # load THIS node's page to get children counts
                depth=depth + 1,
            )
            for i, child in enumerate(children_in_tree)
        ]
        result["children"] = await asyncio.gather(*child_tasks)

        # true_count = sum of children (corrects capped parents like Arbeitsspeicher)
        child_sum = sum(
            c["true_count"] for c in result["children"]
            if c["true_count"] is not None
        )
        result["true_count"] = child_sum if child_sum > 0 else result["reported_count"]

        if result["capped"] and child_sum > 0:
            log.info("%s%-45s  TRUE count=%d  (nav reported %d, was capped)",
                     indent, node["name"], result["true_count"], result["reported_count"])
    else:
        result["true_count"] = result["reported_count"]

    return result


async def run_full_probe(output_csv: str = "jacob_true_counts.csv"):
    sem = asyncio.Semaphore(CONCURRENCY)

    # load /q/ once to get all top-level counts
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True, proxy=pw_proxy(0))
        context = await browser.new_context(locale="de-DE")
        pw_page = await context.new_page()
        await pw_page.route("**/*", lambda route: (
            route.abort() if route.request.resource_type in
            ("image", "media", "font", "stylesheet") else route.continue_()
        ))
        toplevel_counts = await fetch_all_category_counts(pw_page, "https://www.jacob.de/q/")
        await browser.close()
        log.info("Top-level counts from /q/: %s", toplevel_counts)

    # inject top-level counts into tree and probe children
    async with async_playwright() as pw:
        tasks = []
        for i, top in enumerate(TREE):
            key = top["name"].lower().strip()
            top_count = toplevel_counts.get(key)

            # wrap top node with its known count, then drill children
            async def probe_top(top=top, top_count=top_count, slot=i % CONCURRENCY):
                result = {
                    "name":           top["name"],
                    "url":            top["url"],
                    "reported_count": top_count,
                    "true_count":     None,
                    "capped":         top_count is not None and top_count >= CAP_THRESHOLD,
                    "drilled":        False,
                    "children":       [],
                }
                log.info("%-45s  top-level count=%s", top["name"], top_count)

                children_in_tree = top.get("children", [])
                if children_in_tree:
                    result["drilled"] = True
                    child_tasks = [
                        probe_node(
                            pw, sem,
                            slot=(slot + j) % CONCURRENCY,
                            node=child,
                            parent_url=top["url"],
                            depth=1,
                        )
                        for j, child in enumerate(children_in_tree)
                    ]
                    result["children"] = await asyncio.gather(*child_tasks)
                    child_sum = sum(
                        c["true_count"] for c in result["children"]
                        if c["true_count"] is not None
                    )
                    result["true_count"] = child_sum if child_sum > 0 else top_count
                else:
                    result["true_count"] = top_count
                return result

            tasks.append(probe_top())

        all_results = await asyncio.gather(*tasks)

    # ── flatten to CSV ─────────────────────────────────────────────────────────
    rows = []

    def flatten(node, category="", subcategory="", depth=0):
        name = node["name"]
        if depth == 0:
            cat, subcat = name, ""
        elif depth == 1:
            cat, subcat = category, name
        else:
            cat, subcat = category, subcategory

        rows.append({
            "category":       cat,
            "subcategory":    subcat,
            "name":           name,
            "depth":          depth,
            "reported_count": node["reported_count"],
            "true_count":     node["true_count"],
            "was_capped":     "YES" if node["capped"] else "NO",
            "url":            node["url"],
        })
        for child in node.get("children", []):
            flatten(child, cat, subcat if depth >= 1 else name, depth + 1)

    for r in all_results:
        flatten(r)

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "category","subcategory","name","depth",
            "reported_count","true_count","was_capped","url"
        ])
        writer.writeheader()
        writer.writerows(rows)

    # ── top-level summary ──────────────────────────────────────────────────────
    top_rows = [r for r in rows if r["depth"] == 0]
    total    = sum(r["true_count"] or 0 for r in top_rows)

    print(f"\nSaved {output_csv}  ({len(rows)} rows)\n")
    print(f"{'Category':<28} {'Reported':>10}  {'True Count':>12}  Note")
    print("-" * 70)
    for r in sorted(top_rows, key=lambda x: -(x["true_count"] or 0)):
        note = "was capped" if r["was_capped"] == "YES" else ""
        print(f"  {r['name']:<26} {str(r['reported_count'] or '?'):>10}"
              f"  {str(r['true_count'] or '?'):>12}  {note}")
    print(f"\n  {'TOTAL':<26} {'':>10}  {total:>12,}")

asyncio.run(run_full_probe())

01:02:01 [ERROR] Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
Traceback (most recent call last):
  File "/usr/lib/python3/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_257915/1911810220.py", line 199, in <module>
    asyncio.run(run_full_probe())
  File "/usr/lib/python3/dist-packages/nest_asyncio.py", line 38, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3/dist-packages/nest_asyncio.py", line 75, in run_until_complete
    self._run_once()
  File "/usr/lib/python3/dist-packages/nest_asyncio.py", line 98, in _run_once
    event_list = self._selector.select(timeout)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 468, in select
    fd_event_list = self._selector.poll(timeout, ma


Saved jacob_true_counts.csv  (648 rows)

Category                       Reported    True Count  Note
----------------------------------------------------------------------
  PC & Zubehör                   416125         62256  was capped
  Enterprise                      53135         51874  was capped
  Netzwerk                        51399         50608  was capped
  Haushalt                        40822         40585  was capped
  Smartphone & Foto                9335          8151  
  Notebooks & Tablets              9265          7848  
  Office                           5935          5988  
  TV & Audio                       5216          2945  
  Apple                            1116          1048  

  TOTAL                                       231,303


In [6]:
import asyncio
import json
import logging
import random
import re
import time
from urllib.parse import urljoin, urlparse

import nest_asyncio
nest_asyncio.apply()

from curl_cffi.requests import AsyncSession
from selectolax.parser import HTMLParser

# ── logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
logging.getLogger("curl_cffi").setLevel(logging.WARNING)

# ── proxy ─────────────────────────────────────────────────────────────────────
PROXY_HOST = "brd.superproxy.io"
PROXY_PORT = 22225
PROXY_USER = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS = "ymg5cg3a1z33"

def make_proxies():
    token = f"{random.random():.16f}"
    user = f"{PROXY_USER}-session-{token}"
    url = f"http://{user}:{PROXY_PASS}@{PROXY_HOST}:{PROXY_PORT}"
    return {"http": url, "https": url}

# ── config ────────────────────────────────────────────────────────────────────
BASE_URL = "https://www.jacob.de"
SORT = "preis_up"
MAX_PAGE = 500
CONCURRENCY = 5

CATEGORY_URL = "https://www.jacob.de/apple/"
OUTPUT_FILE = "jacob_single_category_pdp.json"
PRODUCT_CAP = 100  # set to None for full run

HEADERS = {
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "de-DE,de;q=0.9,en;q=0.8",
    "referer": "https://www.google.com/",
}

# ── fetch ─────────────────────────────────────────────────────────────────────
async def fetch(session: AsyncSession, url: str, retries: int = 3) -> tuple[str, int, float]:
    last_err = None
    for attempt in range(retries + 1):
        t0 = time.perf_counter()
        try:
            r = await session.get(
                url,
                headers=HEADERS,
                proxies=make_proxies(),
                timeout=45,
            )
            elapsed = time.perf_counter() - t0
            if r.status_code in (429, 500, 502, 503, 504):
                raise Exception(f"retryable {r.status_code}")
            r.raise_for_status()
            log.info("← %d  %5.1fs  %7d B  %s", r.status_code, elapsed, len(r.content), url)
            return r.text, len(r.content), elapsed
        except Exception as e:
            elapsed = time.perf_counter() - t0
            log.warning("attempt %d FAILED  %.1fs  %s  %s", attempt + 1, elapsed, type(e).__name__, url)
            last_err = e
            if attempt < retries:
                await asyncio.sleep(1.5 * (2 ** attempt))
    raise last_err

# ── helpers ───────────────────────────────────────────────────────────────────
def normalize_url(url: str) -> str:
    p = urlparse(url)
    return f"{p.scheme}://{p.netloc}{p.path.rstrip('/')}/"

def build_plp_url(cat_url: str, min_p: int, max_p: int, page: int) -> str:
    return f"{cat_url}?sortBy={SORT}&price-min={min_p}&price-max={max_p}&page={page}"

def clean_price_string(text: str) -> str | None:
    if not text:
        return None
    m = re.search(r"(\d[\d\.\s]*,\d{2}|\d[\d\.\s]*\.\d{2})", text)
    if not m:
        return None
    raw = m.group(1).replace(" ", "")
    if "," in raw and "." in raw:
        raw = raw.replace(".", "").replace(",", ".")
    elif "," in raw:
        raw = raw.replace(".", "").replace(",", ".")
    return raw

def collapse_spaces(text: str | None) -> str | None:
    if not text:
        return None
    return re.sub(r"\s+", " ", text).strip()

def safe_float(x):
    try:
        return float(str(x))
    except Exception:
        return None

def normalize_condition(label: str) -> str:
    x = re.sub(r"\s+", " ", (label or "").strip()).lower()
    mapping = {
        "new": "New",
        "neu": "New",
        "brand new": "New",
        "refurbished": "Refurbished",
        "like new": "Refurbished",
        "renewed": "Refurbished",
        "gebraucht": "Refurbished",
        "b-ware": "Refurbished",
        "b ware": "Refurbished",
        "used": "Refurbished",
        "geöffnet": "Refurbished",
        "geoeffnet": "Refurbished",
        "open box": "Refurbished",
        "outlet": "Refurbished",
        "neuwertig": "Refurbished",
    }
    return mapping.get(x, label.strip() if label else "")

def infer_condition_from_text(*values: str) -> str:
    hay = " ".join(v or "" for v in values).lower()
    refurb_clues = [
        "geöffnet", "geoeffnet", "outlet", "b-ware", "b ware", "gebraucht",
        "used", "refurbished", "like new", "renewed", "open box", "neuwertig",
    ]
    return "Refurbished" if any(c in hay for c in refurb_clues) else "New"

def pick_best_text(values: list[str | None]) -> str | None:
    vals = [v.strip() for v in values if isinstance(v, str) and v.strip()]
    if not vals:
        return None
    vals.sort(key=len, reverse=True)
    return vals[0]

# ── PLP ───────────────────────────────────────────────────────────────────────
def plp_has_products(html: str) -> bool:
    return bool(HTMLParser(html).css("a[href*='/produkte/']"))

def extract_plp_items(html: str) -> list[dict]:
    tree = HTMLParser(html)
    items = {}

    for a in tree.css("a[href*='/produkte/']"):
        href = a.attributes.get("href", "")
        url = href if href.startswith("http") else urljoin(BASE_URL, href)

        artnr_m = re.search(r"artnr-(\d+)", url)
        artnr = artnr_m.group(1) if artnr_m else None

        name = None
        price = None
        currency = None

        card = a
        for _ in range(8):
            if not card:
                break
            if card.css_first("div.c1_price") or card.css_first("input[name='price']"):
                break
            card = card.parent

        if card:
            title_node = card.css_first("a.product_price_link")
            if title_node:
                title = title_node.attributes.get("title", "").strip()
                name = title or None

            hidden_price = card.css_first("input[name='price']")
            if hidden_price:
                price = hidden_price.attributes.get("value")
                currency = "EUR" if price else None

            if not price:
                price_node = card.css_first("div.c1_price")
                if price_node:
                    price = clean_price_string(price_node.text(separator=" ", strip=True))
                    currency = "EUR" if price else None

        if url not in items:
            items[url] = {
                "url": url,
                "artnr": artnr,
                "name": name,
                "price": price,
                "currency": currency,
            }

    return list(items.values())

async def find_price_ceiling(session: AsyncSession, cat_url: str, sem: asyncio.Semaphore) -> int:
    high = 10_000
    while high <= 2_000_000:
        async with sem:
            html, _, _ = await fetch(session, build_plp_url(cat_url, 1, high, 1))
        if not plp_has_products(html):
            break
        high *= 2
    log.info("price ceiling for %s: %d", cat_url, high)
    return high

async def split_ranges(
    session: AsyncSession,
    cat_url: str,
    min_p: int,
    max_p: int,
    sem: asyncio.Semaphore,
    depth: int = 0,
    max_depth: int = 20,
) -> list[tuple[int, int]]:
    if depth > max_depth or min_p >= max_p - 1:
        return [(min_p, max_p)]
    async with sem:
        html, _, _ = await fetch(session, build_plp_url(cat_url, min_p, max_p, MAX_PAGE))
    if not plp_has_products(html):
        return [(min_p, max_p)]
    mid = max(min_p + 1, (min_p + max_p) // 2)
    left, right = await asyncio.gather(
        split_ranges(session, cat_url, min_p, mid, sem, depth + 1),
        split_ranges(session, cat_url, mid, max_p, sem, depth + 1),
    )
    return left + right

async def collect_category(session: AsyncSession, cat_url: str, sem: asyncio.Semaphore) -> tuple[list[dict], int, int]:
    ceiling = await find_price_ceiling(session, cat_url, sem)

    async with sem:
        html, _, _ = await fetch(session, build_plp_url(cat_url, 1, ceiling, MAX_PAGE))

    ranges = await split_ranges(session, cat_url, 1, ceiling, sem) if plp_has_products(html) else [(1, ceiling)]
    log.info("category=%s  ranges=%d", cat_url, len(ranges))

    all_items = []
    seen = set()
    total_reqs = 0
    total_bytes = 0

    for min_p, max_p in ranges:
        page = 1
        while True:
            async with sem:
                html, nb, _ = await fetch(session, build_plp_url(cat_url, min_p, max_p, page))
            total_reqs += 1
            total_bytes += nb

            items = extract_plp_items(html)
            if not items:
                break

            new = [it for it in items if it["url"] not in seen]
            seen |= {it["url"] for it in new}
            all_items.extend(new)
            log.info("  range=%d-%d  page=%d  new=%d  total=%d", min_p, max_p, page, len(new), len(all_items))

            if PRODUCT_CAP is not None and len(all_items) >= PRODUCT_CAP:
                all_items = all_items[:PRODUCT_CAP]
                log.info("product cap reached: %d", PRODUCT_CAP)
                return all_items, total_reqs, total_bytes

            page += 1

    return all_items, total_reqs, total_bytes

# ── PDP parsing ───────────────────────────────────────────────────────────────
def extract_jsonld_product(html: str) -> dict | None:
    blocks = re.findall(
        r'<script[^>]+type=["\']application/ld\+json["\'][^>]*>(.*?)</script>',
        html,
        flags=re.S | re.I,
    )
    for block in blocks:
        try:
            obj = json.loads(block.strip())
        except json.JSONDecodeError:
            continue
        items = obj if isinstance(obj, list) else obj.get("@graph", [obj])
        for item in items:
            if isinstance(item, dict) and item.get("@type") == "Product":
                return item
    return None

def extract_offer_price(product: dict) -> tuple[str | None, str | None, str | None]:
    offers = product.get("offers") or {}

    if isinstance(offers, list):
        values = []
        currency = None
        availability = None
        for off in offers:
            if not isinstance(off, dict):
                continue
            p = off.get("price") or off.get("highPrice") or off.get("lowPrice")
            if p is not None:
                try:
                    values.append(float(str(p)))
                except Exception:
                    pass
            currency = currency or off.get("priceCurrency")
            raw_av = off.get("availability", "")
            if raw_av and not availability:
                availability = raw_av.split("/")[-1]
        return (f"{max(values):.2f}" if values else None, currency, availability)

    if isinstance(offers, dict):
        currency = offers.get("priceCurrency")
        raw_av = offers.get("availability", "")
        availability = raw_av.split("/")[-1] if raw_av else None
        if offers.get("@type") == "AggregateOffer":
            high = offers.get("highPrice")
            low = offers.get("lowPrice")
            price = high if high is not None else low
            return (str(price) if price is not None else None, currency, availability)
        price = offers.get("price")
        return (str(price) if price is not None else None, currency, availability)

    return None, None, None

def extract_html_header_meta(tree: HTMLParser, name: str | None) -> dict:
    text = tree.text(separator=" ", strip=True)

    brand = None
    artnr = None
    ean = None
    mpn = None

    artnr_m = re.search(r"ArtNr\s*:?\s*(\d+)", text, flags=re.I)
    if artnr_m:
        artnr = artnr_m.group(1)

    ean_m = re.search(r"GTIN\s*:?\s*(\d{8,14})", text, flags=re.I)
    if ean_m:
        ean = ean_m.group(1)

    top_line_m = re.search(r"([A-Za-z0-9&\-\.\s]+)\s*\(([A-Z0-9\/\-\._]+)\)\s*ArtNr", text)
    if top_line_m:
        maybe_brand = collapse_spaces(top_line_m.group(1))
        maybe_mpn = top_line_m.group(2)
        if maybe_brand and len(maybe_brand.split()) <= 3:
            brand = maybe_brand
        mpn = maybe_mpn

    if not mpn and name:
        mpn_m = re.findall(r"\(([A-Z0-9\/\-\._]+)\)", name)
        mpn = next((x for x in mpn_m if len(x) >= 4), None)

    return {
        "brand": brand,
        "artnr": artnr,
        "ean": ean,
        "mpn": mpn,
    }

def extract_html_description(tree: HTMLParser) -> str | None:
    box = tree.css_first("div.c1_productDesc")
    if not box:
        return None
    parts = []
    for node in box.css("p"):
        txt = collapse_spaces(node.text(separator=" ", strip=True))
        if txt:
            parts.append(txt)
    if parts:
        return " ".join(parts)
    txt = collapse_spaces(box.text(separator=" ", strip=True))
    if not txt:
        return None
    return re.sub(r"^Product description\s*", "", txt, flags=re.I)

def extract_html_visible_price(tree: HTMLParser) -> str | None:
    candidates = [
        ".c1_product__offer .product__price",
        ".c1_product__offer .c1_price",
        ".product__price__wrapper",
        ".product__price",
        ".c1_price",
    ]
    for sel in candidates:
        for node in tree.css(sel):
            txt = node.text(separator=" ", strip=True)
            price = clean_price_string(txt)
            if price:
                return price
    return None

def extract_html_stock_delivery_availability(tree: HTMLParser) -> tuple[int | None, str | None, str | None]:
    text = tree.text(separator=" ", strip=True)

    delivery_time = None
    stock = None
    availability = None

    blocks = tree.css(".productInfo__benefits")
    for block in blocks:
        block_text = collapse_spaces(block.text(separator=" ", strip=True))
        if not block_text:
            continue

        delivery_match = re.search(
            r"(Lieferzeit\s*[^.]*?Werktage|Sofort lieferbar(?:\s*\(auf Lager\))?|Nicht lieferbar|Vorbestellbar)",
            block_text,
            flags=re.I,
        )
        if delivery_match:
            delivery_time = collapse_spaces(delivery_match.group(1))

        qty_match = re.search(
            r"(?:Only\s+(\d+)\s+pieces\s+left|Nur\s+noch\s+(\d+)\s+St[uü]ck\s+verf[uü]gbar|Noch\s+(\d+)\s+St[uü]ck\s+verf[uü]gbar)",
            block_text,
            flags=re.I,
        )
        if qty_match:
            for idx in (1, 2, 3):
                val = qty_match.group(idx)
                if val:
                    stock = int(val)
                    break

        if "sofort lieferbar" in block_text.lower() or "lieferzeit" in block_text.lower():
            availability = "InStock"
        elif "nicht lieferbar" in block_text.lower():
            availability = "OutOfStock"
        elif "vorbestell" in block_text.lower():
            availability = "PreOrder"

        if delivery_time or stock is not None or availability:
            break

    if not availability:
        lower = text.lower()
        if "sofort lieferbar" in lower or "lieferzeit" in lower or "verfügbar" in lower:
            availability = "InStock"
        elif "nicht lieferbar" in lower:
            availability = "OutOfStock"
        elif "vorbestell" in lower:
            availability = "PreOrder"

    return stock, delivery_time, availability

def extract_condition_options(html: str, current_url: str) -> list[dict]:
    tree = HTMLParser(html)
    rows = []

    for li in tree.css("li.variationItem"):
        a = li.css_first("a[href]")
        container = a or li

        blocks = [x.text(strip=True) for x in container.css("div") if x.text(strip=True)]
        if len(blocks) < 2:
            text = container.text(separator=" ", strip=True)
            lines = [t.strip() for t in re.split(r"\n+", text) if t.strip()]
            if len(lines) >= 2:
                blocks = lines

        if len(blocks) < 2:
            continue

        raw_condition = blocks[0]
        raw_price = blocks[1]
        price = clean_price_string(raw_price)
        if not price:
            continue

        href = a.attributes.get("href") if a else None
        item_url = urljoin(BASE_URL, href) if href else current_url

        rows.append({
            "condition": normalize_condition(raw_condition),
            "raw_condition": raw_condition,
            "price": price,
            "currency": "EUR",
            "url": item_url,
            "is_current": a is None,
        })

    # if duplicate same url/price but raw label differs, prefer normalized labels
    dedup = {}
    for row in rows:
        key = (row["condition"], row["url"])
        existing = dedup.get(key)
        if not existing:
            dedup[key] = row
        else:
            existing_price = safe_float(existing["price"]) or 0
            new_price = safe_float(row["price"]) or 0
            if existing_price == 0 or new_price > 0:
                dedup[key] = row

    rows = list(dedup.values())
    rows.sort(key=lambda r: (0 if r["condition"] == "New" else 1, r["url"]))
    return rows

def parse_single_pdp(html: str, url: str, plp_data: dict) -> dict:
    tree = HTMLParser(html)
    product = extract_jsonld_product(html)
    condition_options = extract_condition_options(html, url)

    jsonld_brand = None
    jsonld_sku = None
    jsonld_mpn = None
    jsonld_ean = None
    jsonld_desc = None
    jsonld_name = None
    jsonld_cat_path = []
    jsonld_price = None
    jsonld_currency = None
    jsonld_availability = None

    if product:
        brand_raw = product.get("brand")
        jsonld_brand = (
            brand_raw.get("name") if isinstance(brand_raw, dict)
            else brand_raw if isinstance(brand_raw, str)
            else None
        )
        jsonld_sku = product.get("sku")
        jsonld_mpn = product.get("mpn")
        jsonld_ean = product.get("gtin13") or product.get("gtin14")
        jsonld_desc = product.get("description")
        jsonld_name = product.get("name")
        jsonld_price, jsonld_currency, jsonld_availability = extract_offer_price(product)

        cat_raw = product.get("category")
        if isinstance(cat_raw, str) and cat_raw.strip():
            jsonld_cat_path = [c.strip() for c in cat_raw.split("/")]

    html_meta = extract_html_header_meta(tree, jsonld_name or plp_data.get("name"))
    html_desc = extract_html_description(tree)
    html_price = extract_html_visible_price(tree)
    html_stock, html_delivery_time, html_availability = extract_html_stock_delivery_availability(tree)

    name = pick_best_text([jsonld_name, plp_data.get("name")])
    brand = pick_best_text([jsonld_brand, html_meta.get("brand")])
    sku = pick_best_text([jsonld_sku, plp_data.get("artnr"), html_meta.get("artnr")])
    mpn = pick_best_text([jsonld_mpn, html_meta.get("mpn")])
    ean = pick_best_text([jsonld_ean, html_meta.get("ean")])
    description = pick_best_text([jsonld_desc, html_desc])
    price = pick_best_text([html_price, jsonld_price, plp_data.get("price")])
    currency = pick_best_text([jsonld_currency, plp_data.get("currency"), "EUR"])
    availability = pick_best_text([jsonld_availability, html_availability])
    category_path = jsonld_cat_path or []

    inferred_condition = infer_condition_from_text(name, description, " ".join(category_path), url)

    current_option = {
        "condition": inferred_condition,
        "raw_condition": "current",
        "price": price or "",
        "currency": currency or "EUR",
        "url": url,
        "is_current": True,
        "delivery_time": html_delivery_time or "",
        "stock": html_stock,
        "availability": availability,
    }

    return {
        "url": url,
        "artnr": pick_best_text([plp_data.get("artnr"), html_meta.get("artnr"), sku]),
        "name": name,
        "brand": brand,
        "sku": sku,
        "mpn": mpn,
        "ean": ean,
        "description": description,
        "currency": currency or "EUR",
        "availability": availability,
        "category_path": category_path,
        "current_option": current_option,
        "variation_options": condition_options,
    }

def canonical_key(row: dict) -> str:
    if row.get("ean"):
        return f"ean:{row['ean']}"
    if row.get("mpn"):
        return f"mpn:{row['mpn']}"
    if row.get("sku"):
        return f"sku:{row['sku']}"
    return f"url:{row['url']}"

def merge_two_option_product(base_row: dict, sibling_row: dict | None) -> dict:
    option_map = {}

    for row in [base_row, sibling_row]:
        if not row:
            continue
        opt = row["current_option"]
        cond = normalize_condition(opt["condition"])
        option_map[cond] = {
            "condition": cond,
            "raw_condition": opt.get("raw_condition", "current"),
            "price": opt.get("price", ""),
            "currency": opt.get("currency", "EUR"),
            "url": opt.get("url"),
            "is_current": opt.get("is_current", False),
            "delivery_time": opt.get("delivery_time", ""),
            "stock": opt.get("stock"),
            "availability": opt.get("availability"),
        }

    for opt in base_row.get("variation_options", []):
        cond = normalize_condition(opt["condition"])
        if cond not in option_map:
            option_map[cond] = {
                "condition": cond,
                "raw_condition": opt.get("raw_condition", ""),
                "price": opt.get("price", ""),
                "currency": opt.get("currency", "EUR"),
                "url": opt.get("url"),
                "is_current": opt.get("is_current", False),
                "delivery_time": "",
                "stock": None,
                "availability": None,
            }

    condition_options = list(option_map.values())
    condition_options.sort(key=lambda r: (0 if r["condition"] == "New" else 1, r["url"] or ""))

    price_dict = {"New": "", "Refurbished": ""}
    for opt in condition_options:
        if opt["condition"] in price_dict:
            price_dict[opt["condition"]] = opt.get("price", "")

    merged_rows = [r for r in [base_row, sibling_row] if r]

    return {
        "artnr": pick_best_text([r.get("artnr") for r in merged_rows]),
        "name": pick_best_text([r.get("name") for r in merged_rows]),
        "brand": pick_best_text([r.get("brand") for r in merged_rows]),
        "sku": pick_best_text([r.get("sku") for r in merged_rows]),
        "mpn": pick_best_text([r.get("mpn") for r in merged_rows]),
        "ean": pick_best_text([r.get("ean") for r in merged_rows]),
        "description": pick_best_text([r.get("description") for r in merged_rows]),
        "price": price_dict,
        "currency": pick_best_text([r.get("currency") for r in merged_rows]) or "EUR",
        "availability": pick_best_text([r.get("availability") for r in merged_rows]),
        "category_path": max((r.get("category_path", []) for r in merged_rows), key=lambda x: len(x), default=[]),
        "condition_options": condition_options,
    }

# ── PDP crawl ─────────────────────────────────────────────────────────────────
async def fetch_parsed_pdp(session: AsyncSession, item: dict, sem: asyncio.Semaphore) -> dict | None:
    try:
        async with sem:
            html, _, _ = await fetch(session, item["url"])
        row = parse_single_pdp(html, item["url"], item)
        log.info(
            "PDP: name=%-50s ean=%s cond=%s stock=%s url=%s",
            (row.get("name") or "")[:50],
            row.get("ean"),
            row["current_option"]["condition"],
            row["current_option"]["stock"],
            row.get("url"),
        )
        return row
    except Exception as e:
        log.warning("PDP FAILED  %s  %s  %s", type(e).__name__, item["url"], e)
        return None

async def enrich_product(session: AsyncSession, item: dict, sem: asyncio.Semaphore) -> dict | None:
    base_row = await fetch_parsed_pdp(session, item, sem)
    if not base_row:
        return None

    sibling_url = None
    sibling_price = None
    sibling_currency = None

    for opt in base_row.get("variation_options", []):
        opt_url = opt.get("url")
        if opt_url and opt_url != item["url"]:
            sibling_url = opt_url
            sibling_price = opt.get("price")
            sibling_currency = opt.get("currency")
            break

    sibling_row = None
    if sibling_url:
        artnr_m = re.search(r"artnr-(\d+)", sibling_url)
        sibling_stub = {
            "url": sibling_url,
            "artnr": artnr_m.group(1) if artnr_m else None,
            "name": None,
            "price": sibling_price,
            "currency": sibling_currency,
        }
        sibling_row = await fetch_parsed_pdp(session, sibling_stub, sem)

        # ignore unrelated sibling accidentally linked
        if sibling_row and canonical_key(sibling_row) != canonical_key(base_row):
            log.warning("sibling key mismatch, ignoring sibling: %s", sibling_url)
            sibling_row = None

    return merge_two_option_product(base_row, sibling_row)

# ── runner ────────────────────────────────────────────────────────────────────
async def run():
    sem = asyncio.Semaphore(CONCURRENCY)

    async with AsyncSession(impersonate="chrome124") as session:
        log.info("=== STEP 1: proxy health check ===")
        try:
            _, nb, t = await fetch(session, "https://geo.brdtest.com/welcome.txt")
            log.info("proxy OK  size=%d  time=%.2fs", nb, t)
        except Exception as e:
            log.critical("proxy FAILED — aborting: %s", e)
            return

        log.info("=== STEP 2: PLP collection for one category ===")
        category_url = normalize_url(CATEGORY_URL)
        t0 = time.perf_counter()
        plp_items, plp_reqs, plp_bytes = await collect_category(session, category_url, sem)
        plp_elapsed = time.perf_counter() - t0
        log.info("PLP done  items=%d requests=%d bytes=%.1f KB", len(plp_items), plp_reqs, plp_bytes / 1024)

        log.info("=== STEP 3: PDP enrichment ===")
        results = []
        for idx, item in enumerate(plp_items, 1):
            row = await enrich_product(session, item, sem)
            if row:
                results.append(row)
            log.info("progress: %d/%d", idx, len(plp_items))

        # final dedupe by logical key
        dedup = {}
        for row in results:
            dedup[canonical_key(row)] = row
        final_rows = list(dedup.values())
        final_rows.sort(key=lambda r: ((r.get("brand") or ""), (r.get("name") or ""), (r.get("ean") or "")))

        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(final_rows, f, ensure_ascii=False, indent=2)
        log.info("saved %s  products=%d", OUTPUT_FILE, len(final_rows))

    avg_plp_kb = plp_bytes / max(plp_reqs, 1) / 1024
    avg_plp_t = plp_elapsed / max(plp_reqs, 1)

    print(f"""
╔══════════════════════════════════════════════════════════════════╗
  jacob.de — SINGLE CATEGORY PDP RUN
╚══════════════════════════════════════════════════════════════════╝

CATEGORY
  URL               : {CATEGORY_URL}

STACK
  HTTP client       : curl_cffi chrome124
  Parser            : selectolax (PLP + PDP HTML) | JSON-LD regex (PDP)
  Proxy             : Bright Data datacenter (DC)
  Concurrency       : asyncio.Semaphore({CONCURRENCY})

LIMITS
  Product cap       : {PRODUCT_CAP if PRODUCT_CAP is not None else "FULL RUN"}

PLP
  Products found    : {len(plp_items)}
  Requests made     : {plp_reqs}
  Bytes transferred : {plp_bytes/1024:.1f} KB
  Avg page size     : {avg_plp_kb:.1f} KB
  Avg response time : {avg_plp_t:.2f}s
  Time elapsed      : {plp_elapsed:.1f}s

PDP
  Final merged rows : {len(final_rows)}

DEDUP
  Primary key       : GTIN/EAN
  Fallbacks         : MPN -> SKU -> URL

OUTPUT
  File              : {OUTPUT_FILE}
""")

asyncio.run(run())


17:24:35 [INFO] === STEP 1: proxy health check ===
17:24:36 [INFO] ← 200    0.6s      618 B  https://geo.brdtest.com/welcome.txt
17:24:36 [INFO] proxy OK  size=618  time=0.64s
17:24:36 [INFO] === STEP 2: PLP collection for one category ===
17:24:37 [INFO] ← 200    1.5s   412856 B  https://www.jacob.de/apple/?sortBy=preis_up&price-min=1&price-max=10000&page=1
17:24:38 [INFO] ← 200    0.5s   412856 B  https://www.jacob.de/apple/?sortBy=preis_up&price-min=1&price-max=20000&page=1
17:24:38 [INFO] ← 200    0.4s   412874 B  https://www.jacob.de/apple/?sortBy=preis_up&price-min=1&price-max=40000&page=1
17:24:39 [INFO] ← 200    0.5s   412856 B  https://www.jacob.de/apple/?sortBy=preis_up&price-min=1&price-max=80000&page=1
17:24:39 [INFO] ← 200    0.5s   412866 B  https://www.jacob.de/apple/?sortBy=preis_up&price-min=1&price-max=160000&page=1
17:24:40 [INFO] ← 200    0.6s   412866 B  https://www.jacob.de/apple/?sortBy=preis_up&price-min=1&price-max=320000&page=1
17:24:40 [INFO] ← 200    0.6s   


╔══════════════════════════════════════════════════════════════════╗
  jacob.de — SINGLE CATEGORY PDP RUN
╚══════════════════════════════════════════════════════════════════╝

CATEGORY
  URL               : https://www.jacob.de/apple/

STACK
  HTTP client       : curl_cffi chrome124
  Parser            : selectolax (PLP + PDP HTML) | JSON-LD regex (PDP)
  Proxy             : Bright Data datacenter (DC)
  Concurrency       : asyncio.Semaphore(5)

LIMITS
  Product cap       : 100

PLP
  Products found    : 100
  Requests made     : 5
  Bytes transferred : 2013.9 KB
  Avg page size     : 402.8 KB
  Avg response time : 7.39s
  Time elapsed      : 36.9s

PDP
  Final merged rows : 98

DEDUP
  Primary key       : GTIN/EAN
  Fallbacks         : MPN -> SKU -> URL

OUTPUT
  File              : jacob_single_category_pdp.json

